# 【講師用】Ch.2 データ可視化・EDA 完全解説

> 🔒 **このノートブックは講師専用です。受講者には配布しないでください。**

---

## 📢 講師向け冒頭ガイド

### このチャプターで最も伝えたいこと
- **「グラフを作ること」が目的ではなく「仮説を立てること」が目的**
- グラフのコードは AI に任せてOK。「何を見たいか」「何が読み取れるか」が DS の仕事
- Ch.1 で数字で見えたことがグラフで一目瞭然になる体験をさせる

### よくある受講者の躓きポイント
| 躓き | 対処法 |
|------|--------|
| `plt.subplots` の使い方がわからない | 「`fig, axes = plt.subplots(行数, 列数)` で複数グラフを並べられる。コードは AI に任せてOK」と伝える |
| 相関係数の値の解釈ができない | 「0.7以上は強い相関。ただし相関 ≠ 因果関係」と繰り返す |
| グラフを作っただけで終わってしまう | 「このグラフから何がわかりますか？」と必ず声をかける |

### 時間配分目安（座学20分）
- 2.1 ヒストグラム：4分
- 2.2 箱ひげ図：4分
- 2.3 散布図：4分
- 2.4 相関ヒートマップ：5分
- 2.5 pairplot（デモのみ）：3分

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib  # 日本語フォント対応
from sklearn.datasets import load_wine

# データ準備（Ch.1 と同じ）
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target
df['品種'] = df['target'].map({0: 'Barolo', 1: 'Grignolino', 2: 'Barbera'})

print('データ準備完了:', df.shape)

---
## 2.1 ヒストグラム：データの分布を見る

### 📢 講師メモ
「分布」という言葉が抽象的に感じる受講者には：
> 「クラスの身長データを並べたとき、何人がどの範囲にいるかを棒グラフにしたもの」と説明するとわかりやすい。

品種別の色分けをすることで「品種によって分布が違う = 分類できる可能性がある」ことを視覚的に示します。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左：全体の分布
axes[0].hist(df['alcohol'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('アルコール度数の分布（全体）')
axes[0].set_xlabel('アルコール度数')
axes[0].set_ylabel('件数')

# 右：品種別の分布（alpha=0.6 で半透明にして重なりを見せる）
colors = {'Barolo': '#e74c3c', 'Grignolino': '#2ecc71', 'Barbera': '#3498db'}
for name, group in df.groupby('品種'):
    axes[1].hist(group['alcohol'], bins=15, alpha=0.6,
                 label=name, color=colors[name], edgecolor='white')
axes[1].set_title('アルコール度数の分布（品種別）')
axes[1].set_xlabel('アルコール度数')
axes[1].set_ylabel('件数')
axes[1].legend()

plt.tight_layout()
plt.show()

### 📢 グラフから読み取れること（必ず声に出して言う）
- **全体分布**：右に裾が長い（一部にアルコール度数が高いワインが存在）
- **品種別分布**：
  - **Barolo（赤）**：13〜14.5 に分布。最も右寄り = アルコール高め
  - **Grignolino（緑）**：11.5〜13 に広く分布
  - **Barbera（青）**：12〜13 に集中
- **重要な観察**：Barolo と他の2品種の分布が重なりにくい → アルコール度数だけでも Barolo を識別できそう

> 「このように品種で分布が分かれているということは、機械学習モデルで分類できる根拠があると言えます。」

---
## 2.2 箱ひげ図：品種間の分布を比較する

### 📢 講師メモ
箱ひげ図は「ヒストグラムを圧縮したもの」と説明すると理解が早まります。
5点要約（最小・Q1・中央値・Q3・最大）が一目でわかる優れものです。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

df.boxplot(column='color_intensity', by='品種', ax=ax,
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2),
           whiskerprops=dict(color='steelblue'),
           capprops=dict(color='steelblue'))

ax.set_title('品種ごとの色の濃さ（color_intensity）')
ax.set_xlabel('品種')
ax.set_ylabel('color_intensity')
plt.suptitle('')  # デフォルトの自動タイトルを消す
plt.tight_layout()
plt.show()

### 📢 箱ひげ図の各要素の説明スクリプト

```
【箱の部分】
- 箱の上端 = 第3四分位数（Q3）：上位25%の境界
- 赤い線   = 中央値（Q2）：50%の境界
- 箱の下端 = 第1四分位数（Q1）：下位25%の境界

【ひげの部分】
- ひげの先端 = 外れ値を除いた最大・最小

【点（○）】
- 外れ値（Q3 + 1.5×IQR を超える値）
```

**グラフから読み取れること：**
- Barolo の色が最も濃く（箱の位置が高い）、Grignolino が最も薄い
- Barbera は中程度だが、箱が広い（個体差が大きい）
- Barolo には外れ値（点）が存在 → 特に色が薄い Barolo が一部ある

---
## 2.3 散布図：2変数の関係を見る

### 📢 講師メモ
散布図は「2つの変数が一緒に動くかどうか」を見るグラフです。
品種で色分けすることで「2変数の組み合わせで品種が分離できるか」が一目でわかります。

In [ ]:
colors = {'Barolo': '#e74c3c', 'Grignolino': '#2ecc71', 'Barbera': '#3498db'}

fig, ax = plt.subplots(figsize=(8, 6))
for name, group in df.groupby('品種'):
    ax.scatter(group['alcohol'], group['proline'],
               label=name, color=colors[name], alpha=0.7, s=60)

ax.set_title('アルコール度数 vs プロリン含有量（品種で色分け）')
ax.set_xlabel('アルコール度数')
ax.set_ylabel('プロリン含有量 (proline)')
ax.legend(title='品種')
plt.tight_layout()
plt.show()

### 📢 散布図から読み取れること
- **Barolo（赤）**：右上に集中（アルコール高 + プロリン高）
- **Grignolino（緑）**：左下に集中（アルコール低 + プロリン低）
- **Barbera（青）**：中間付近に散らばる

> 「3品種がほぼ分離できています。つまり『アルコール度数』と『プロリン含有量』の2つだけで品種がかなり判別できることを示しています。
> これが機械学習モデルを使う理由です。人間が目で見るよりも、13個の特徴量をすべて組み合わせて判断できる。」

---
## 2.4 相関ヒートマップ：変数間の相関を一覧で見る

### 📢 講師メモ
相関係数の解釈について、必ず **「相関 ≠ 因果関係」** を強調してください。
よく使う例：「アイスの売り上げと溺死者数は正の相関があるが、アイスが原因ではなく、夏（気温）が共通原因」

In [ ]:
numeric_cols = wine.feature_names
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',  # 赤=正、青=負
            center=0,
            vmin=-1, vmax=1,  # スケールを -1〜1 に固定
            square=True,
            ax=ax)
ax.set_title('特徴量間の相関係数ヒートマップ')
plt.tight_layout()
plt.show()

### 📢 ヒートマップの読み方スクリプト

```
- 対角線（左上〜右下）：自分自身との相関 = 常に 1.0
- 赤色：正の相関（一方が増えると他方も増える傾向）
- 青色：負の相関（一方が増えると他方は減る傾向）
- 白に近い：相関なし
```

**注目ポイント：**
- `total_phenols` と `flavanoids` の相関が高い（約0.86）→ 似た情報を持つ変数
- `alcohol` と `proline` も正の相関 → 散布図で見た「右上に集まる」のと一致
- 相関が高い変数を両方モデルに入れると「多重共線性」が起きる可能性（発展内容）

In [ ]:
# 相関が強い変数の組み合わせを数値で確認
corr_pairs = corr_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs < 1.0]  # 自己相関を除く
corr_pairs = corr_pairs.abs().sort_values(ascending=False)

print('相関が強い変数の組み合わせ（絶対値で上位）：')
print(corr_pairs.head(10)[::2].round(3))  # 重複ペアを除いて上位5組

---
## 2.5 pairplot：全変数ペアの一括確認（デモ）

### 📢 講師メモ
pairplot は実行に時間がかかるため（30秒〜1分）、講師がデモで見せるだけで十分です。
「実務では最初の探索に使うが、変数が多いと画像が小さくなりすぎるため、代表的な変数だけを選ぶ」と伝えましょう。

In [ ]:
# 代表的な4特徴量で pairplot
selected_cols = ['alcohol', 'color_intensity', 'proline', 'flavanoids', '品種']

sns.pairplot(df[selected_cols], hue='品種',
             diag_kind='hist',
             palette={'Barolo': '#e74c3c', 'Grignolino': '#2ecc71', 'Barbera': '#3498db'},
             plot_kws={'alpha': 0.6})
plt.suptitle('主要特徴量のペアプロット', y=1.02, fontsize=14)
plt.show()

### 📢 pairplot の読み方
- **対角線**：各変数のヒストグラム（品種別）
- **対角線以外**：2変数の散布図（品種で色分け）
- 3品種が綺麗に分離している組み合わせ = 分類に使いやすい特徴量の組み合わせ

> 「13変数すべてで pairplot を作ると 13×13=169 個のグラフができます。実務では重要そうな変数に絞ってから使います。」

---
## 🔑 演習の解答・解説

### 📢 演習進行のポイント
- 問1・2はコードをAIで生成させることを明示的に許可する
- **問3の「仮説の言語化」に最も時間をかけてもらう**。グラフを作って終わりにさせない
- 発表時は「何を見たか」ではなく「何がわかったか」を言語化させる

### 問1：棒グラフで品種ごとの平均アルコール度数を比較する

**解説のポイント：** 棒グラフは「グループ間の値を比較する」ときに最もシンプルで伝わりやすいグラフ。

In [ ]:
# ✅ 問1：解答
# 品種ごとの平均アルコール度数を計算
alcohol_mean = df.groupby('品種')['alcohol'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(alcohol_mean.index, alcohol_mean.values,
              color=['#e74c3c', '#3498db', '#2ecc71'],
              edgecolor='white', linewidth=1.5)

# 棒の上に数値を表示（見やすくする）
for bar, val in zip(bars, alcohol_mean.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', va='bottom', fontsize=12)

ax.set_title('品種ごとの平均アルコール度数')
ax.set_xlabel('品種')
ax.set_ylabel('平均アルコール度数')
ax.set_ylim(0, 16)
plt.tight_layout()
plt.show()

**📢 解説：**
- Barolo（約13.74）> Barbera（約12.25）> Grignolino（約12.28）の順
- Barolo が約 1.5 高い = ヒストグラムで見た「右寄り」と一致
- **棒グラフを使う理由**：「3品種のアルコール度数を比較したい」という目的に最適
  - ヒストグラムは分布の形を見るもの。「比較」なら棒グラフが明快

### 問2：自分で選んだ2つの特徴量の散布図を作る

**解説のポイント：** 「なぜその変数を選んだか」の理由を言語化させることが重要。ヒートマップで相関が高い変数を選ぶのが正攻法。

In [ ]:
# ✅ 問2：解答例（total_phenols と flavanoids を選んだ場合）
# 相関係数が約 0.86 と高い組み合わせ

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 例1：相関が高い組み合わせ（total_phenols × flavanoids）
for name, group in df.groupby('品種'):
    axes[0].scatter(group['total_phenols'], group['flavanoids'],
                    label=name, color=colors[name], alpha=0.7, s=60)
axes[0].set_title('total_phenols × flavanoids\n（相関係数 ≈ 0.86）')
axes[0].set_xlabel('total_phenols')
axes[0].set_ylabel('flavanoids')
axes[0].legend(title='品種')

# 例2：相関が低い組み合わせ（alcohol × ash）
for name, group in df.groupby('品種'):
    axes[1].scatter(group['alcohol'], group['ash'],
                    label=name, color=colors[name], alpha=0.7, s=60)
axes[1].set_title('alcohol × ash\n（相関係数 ≈ 0.27）')
axes[1].set_xlabel('alcohol')
axes[1].set_ylabel('ash')
axes[1].legend(title='品種')

plt.tight_layout()
plt.show()

**📢 解説：**
- **左（相関高い）**：total_phenols と flavanoids は一緒に増減する。ほぼ直線状に並ぶ。品種もある程度分離している
- **右（相関低い）**：alcohol と ash はバラバラに散らばる。品種の分離もしにくい
- **重要な教訓**：相関が高い変数の組み合わせは「分類に使いやすい可能性がある」が、情報の重複も生まれる

### 問3：グラフから「仮説」を1つ言語化する

**解説のポイント：** このチャプターで最も重要な問題。受講者に発表させましょう。

**📢 仮説の例（受講者への参考例として使う）：**

```
仮説例1：
「アルコール度数（alcohol）とプロリン含有量（proline）は、
  Barolo を他の品種から区別するのに最も効く特徴量だと思う。
  散布図で Barolo が右上に明確に分離していたから。」

仮説例2：
「total_phenols と flavanoids の相関が高いので、
  両方をモデルに入れても似た情報になる。
  どちらか一方で十分かもしれない。」

仮説例3：
「Grignolino と Barbera は似た傾向が多く、
  機械学習モデルでもこの2品種の区別が最も難しいと思う。」
```

> **Ch.4 への橋渡し：**「この仮説が正しいかどうか、Ch.4 の特徴量重要度を見て確認しましょう。」

### 問4（発展）：箱ひげ図を複数特徴量で並べて比較する

In [ ]:
# ✅ 問4：解答
feature_list = ['alcohol', 'color_intensity', 'proline', 'flavanoids']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.flat, feature_list):
    df.boxplot(column=col, by='品種', ax=ax,
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(col)
    ax.set_xlabel('品種')
    ax.set_ylabel('値')
    plt.sca(ax)
    plt.title(col)

plt.suptitle('主要特徴量の品種別分布', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**📢 解説：**
- `alcohol`：Barolo が最も高く、分離がはっきりしている
- `color_intensity`：Barolo が最も濃い（箱の位置が高い）。外れ値あり
- `proline`：Barolo と他2品種の差が最も大きい = **最も判別力が高い特徴量の候補**
- `flavanoids`：Grignolino・Barbera が低め。Barolo は高め

> 「proline の箱ひげ図を見ると、Barolo の箱が他の品種と全く重なっていません。これは Ch.4 の特徴量重要度で proline が上位に来ることの根拠になります。」

---
## ✅ チャプターのまとめ（講師用コメント付き）

| グラフ | 目的 | コード | 伝えるポイント |
|--------|------|--------|---------------|
| ヒストグラム | 分布の形を見る | `plt.hist()` | 山の位置と形で「どんなデータか」がわかる |
| 箱ひげ図 | グループ間の比較 | `df.boxplot()` | 中央値・分布範囲・外れ値を一目で |
| 散布図 | 2変数の関係 | `plt.scatter()` | 品種で色分けすると分離が見える |
| ヒートマップ | 相関を一覧で | `sns.heatmap()` | 相関 ≠ 因果。高い相関は情報の重複かも |
| ペアプロット | 全変数を一括確認 | `sns.pairplot()` | 探索用。変数が多いと使いにくい |

### 📢 Ch.3 への橋渡し
> 「これまで数値データ（表形式）を扱ってきましたが、次は『画像データ』を扱います。
> 実は画像も数値の集まりです。それを体験してもらいます。」